<a href="https://colab.research.google.com/github/evan-dg31/rag/blob/main/Food_Traceability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG using Chromadb
- In-memory ChromaDB
- No persistence
- No LangChain or other frameworks
- Few sample documents
- Chroma's default embedding model automatically: onnx_models/**all-MiniLM-L6-v2**

## README: How this RAG System Works

This notebook demonstrates a basic Retrieval Augmented Generation (RAG) system using ChromaDB for document storage and retrieval. It's an in-memory setup without external frameworks like LangChain, focusing on the core mechanics.

#### 1. Document Loading

**Mechanism:** Sample text documents related to food traceability are defined directly within the notebook as a Python list.

**Code Reference:** See 'Step 3: Load Sample Documents' cell (`AMIcFUz-Qj_l`).

#### 2. Tokenization and Embedding

**Mechanism:** When documents are added to the ChromaDB collection, the database automatically handles their tokenization and generates numerical representations called embeddings. ChromaDB uses its default embedding model (e.g., `onnx_models/all-MiniLM-L6-v2`) for this process. The text is first broken down into smaller units (tokens), and then these tokens are processed by the embedding model to produce a dense vector that captures the semantic meaning of the text.

**Code Reference:** The `collection.add()` call in 'Step 3: Load Sample Documents' (`AMIcFUz-Qj_l`) triggers this process internally within ChromaDB.

#### 3. Query Embedding

**Mechanism:** When a user poses a question (query), ChromaDB takes this query and also converts it into an embedding using the *same embedding model* that was used for the stored documents. This ensures that the query and document embeddings exist in the same vector space, allowing for meaningful similarity comparisons.

**Code Reference:** The `collection.query(query_texts=[x])` call in 'Step 4: Test Retrieval' (`GBqtgUaRSpue`) performs this embedding of the user's question (`x`).

#### 4. Retrieval

**Mechanism:** After the query is embedded, ChromaDB compares its embedding vector to the embedding vectors of all the documents in the collection. It identifies the documents whose embeddings are most 'similar' (i.e., closest in vector space) to the query's embedding. This similarity is typically measured using distance metrics like cosine similarity.

**Code Reference:** The `collection.query(n_results=2)` in 'Step 4: Test Retrieval' (`GBqtgUaRSpue`) retrieves the top 2 most relevant documents.

#### 5. Context Creation

**Mechanism:** The retrieved relevant documents are then concatenated into a single string, forming a 'context.' This context is intended to provide the necessary background information for an LLM to answer the user's question accurately and informatively.

**Code Reference:** The `context = "\n".join(results["documents"][0])` in 'Step 5: Create Context' (`54llEi0KTMdY`) combines the retrieved documents into a single `context` string.

#### 6. Question Answering (LLM Integration - *Not yet implemented in this notebook*)

**Mechanism:** In a complete RAG system, this generated `context` would be passed along with the original user query to a Large Language Model (LLM). The LLM would then use this context to formulate a coherent and accurate answer to the question, drawing upon the information provided by the retrieved documents. This approach helps the LLM provide factual answers and reduces the likelihood of 'hallucinations' by grounding its responses in specific, retrieved information.

**Next Steps:** To complete the RAG flow, you would integrate an LLM API (e.g., OpenAI, Gemini, Llama 2) to take the `context` and user's `question` to generate a final answer.

## Step 1: Install Dependencies

In [ ]:
!pip install chromadb openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.8 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found 

## Step 2: Create In-Memory ChromaDB

In [ ]:
import chromadb

# In-memory database
client = chromadb.Client()

# Create a collection
collection = client.create_collection(name="food_traceability")

print("Collection created")

Collection created


## Step 3: Load Sample Documents

In [ ]:
documents = [
    """
    Food traceability tracks products from farm to consumer.
    Every product should have a unique lot identifier.
    """,

    """
    Suppliers provide shipment information and product metadata.
    Lot numbers help identify affected products during recalls.
    """,

    """
    Regulatory agencies require food manufacturers to maintain
    records for product tracking and compliance.
    """
]

ids = [
    "doc1",
    "doc2",
    "doc3"
]

collection.add(
    documents=documents,
    ids=ids
)

print("Documents indexed")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 53.2MiB/s]


Documents indexed


## Step 4: Test Retrieval

In [ ]:
query = "How are recalls performed"
results = collection.query(
    query_texts=[query],
    n_results=2
)

results['documents'][0]

['\n    Suppliers provide shipment information and product metadata.\n    Lot numbers help identify affected products during recalls.\n    ',
 '\n    Regulatory agencies require food manufacturers to maintain\n    records for product tracking and compliance.\n    ']

In [ ]:
for i in range(len(results['ids'][0])):
    print(f"ID: {results['ids'][0][i]}, \n Document: {results['documents'][0][i]}")

ID: doc2, 
 Document: 
    Suppliers provide shipment information and product metadata.
    Lot numbers help identify affected products during recalls.
    
ID: doc3, 
 Document: 
    Regulatory agencies require food manufacturers to maintain
    records for product tracking and compliance.
    


## Step 5: Create Context

In [ ]:
context_parts = []

for i in range(len(results['ids'][0])):
    context_parts.append(f"ID: {results['ids'][0][i]}\nDocument: {results['documents'][0][i]}")

context = "\n".join(context_parts)

print(context)

ID: doc2
Document: 
    Suppliers provide shipment information and product metadata.
    Lot numbers help identify affected products during recalls.
    
ID: doc3
Document: 
    Regulatory agencies require food manufacturers to maintain
    records for product tracking and compliance.
    


## Step 6: Add more documents
- Harcode and recreate the document list
- Expand the collection


### Recreate the Document List

In [ ]:
documents = [
    """
    Food traceability tracks products from farm to consumer.
    Every product should have a unique lot identifier.
    """,

    """
    Suppliers provide shipment information and product metadata.
    Lot numbers help identify affected products during recalls.
    """,

    """
    Regulatory agencies require food manufacturers to maintain
    records for product tracking and compliance.
    """,

    """
    Distribution centers receive products from suppliers and
    distribute them to retail stores.
    """,

    """
    Product recalls require identifying all shipments associated
    with an affected lot number.
    """
]

ids = [f"doc{i}" for i in range(1, len(documents)+1)]

collection.add(
    documents=documents,
    ids=ids
)

In [ ]:
collection.get(ids=ids, include=['documents', 'metadatas'])


{'ids': ['doc1', 'doc2', 'doc3', 'doc4', 'doc5'],
 'embeddings': None,
 'documents': ['\n    Food traceability tracks products from farm to consumer.\n    Every product should have a unique lot identifier.\n    ',
  '\n    Suppliers provide shipment information and product metadata.\n    Lot numbers help identify affected products during recalls.\n    ',
  '\n    Regulatory agencies require food manufacturers to maintain\n    records for product tracking and compliance.\n    ',
  '\n    Distribution centers receive products from suppliers and\n    distribute them to retail stores.\n    ',
  '\n    Product recalls require identifying all shipments associated\n    with an affected lot number.\n    '],
 'uris': None,
 'included': ['documents', 'metadatas'],
 'data': None,
 'metadatas': [None, None, None, None, None]}

In [ ]:
collection.get(ids=ids, include=['documents', 'metadatas'])

{'ids': ['doc1', 'doc2', 'doc3', 'doc4', 'doc5'],
 'embeddings': None,
 'documents': ['\n    Food traceability tracks products from farm to consumer.\n    Every product should have a unique lot identifier.\n    ',
  '\n    Suppliers provide shipment information and product metadata.\n    Lot numbers help identify affected products during recalls.\n    ',
  '\n    Regulatory agencies require food manufacturers to maintain\n    records for product tracking and compliance.\n    ',
  '\n    Distribution centers receive products from suppliers and\n    distribute them to retail stores.\n    ',
  '\n    Product recalls require identifying all shipments associated\n    with an affected lot number.\n    '],
 'uris': None,
 'included': ['documents', 'metadatas'],
 'data': None,
 'metadatas': [None, None, None, None, None]}

In [ ]:
query = "How are recalls performed"
results = collection.query(
    query_texts=[query],
    n_results=2
)

context = "\n".join(
    results["documents"][0]
)

print(context)


    Product recalls require identifying all shipments associated
    with an affected lot number.
    

    Suppliers provide shipment information and product metadata.
    Lot numbers help identify affected products during recalls.
    


### Expand the Collection
- You can continuously expand the collection:

In [ ]:
collection.add(
    ids=['doc6', 'doc7'],
    documents = [
        "Retail stores scan products to maintain inventory tracking.",
        "Temperature monitoring ensures food safety during transportation."
    ]
)
print("Documents doc6 and doc7 added.")
print("\nVerifying newly added documents:")
print(collection.get(ids=['doc6', 'doc7'], include=['documents', 'metadatas']))

Documents doc6 and doc7 added.

Verifying newly added documents:
{'ids': ['doc6', 'doc7'], 'embeddings': None, 'documents': ['Retail stores scan products to maintain inventory tracking.', 'Temperature monitoring ensures food safety during transportation.'], 'uris': None, 'included': ['documents', 'metadatas'], 'data': None, 'metadatas': [None, None]}


### Add Metadata
- This is useful for demonstrating enterprise RAG.

In [ ]:
collection.add(
    ids=["doc8"],
    documents=[
        "Milk shipments must be maintained below 40°F during transport."
    ],
    metadatas=[
        {
            "source": "food_safety_guide.pdf",
            "domain": "cold_chain",
            "category": "temperature"
        }
    ]
)

In [ ]:
results = collection.query(
    query_texts=["temperature requirements for milk"],
    where={"source": "food_safety_guide.pdf"},
    n_results=3
)

context = "\n".join(
    results["documents"][0]
)

context

'Milk shipments must be maintained below 40°F during transport.'